In [2]:
"""
LICENSE MIT
2021
Guillaume Rozier
Website : http://www.covidtracker.fr
Mail : guillaume.rozier@telecomnancy.net

README:
This file contains scripts that download data from data.gouv.fr and then process it to build many graphes.
I'm currently cleaning the code, please ask me if something is not clear enough.

The charts are exported to 'charts/images/france'.
Data is download to/imported from 'data/france'.
Requirements: please see the imports below (use pip3 to install them).

"""

"\nLICENSE MIT\n2021\nGuillaume Rozier\nWebsite : http://www.covidtracker.fr\nMail : guillaume.rozier@telecomnancy.net\n\nREADME:\nThis file contains scripts that download data from data.gouv.fr and then process it to build many graphes.\nI'm currently cleaning the code, please ask me if something is not clear enough.\n\nThe charts are exported to 'charts/images/france'.\nData is download to/imported from 'data/france'.\nRequirements: please see the imports below (use pip3 to install them).\n\n"

In [3]:
import pandas as pd
import json
import france_data_management as data

show_charts = False
PATH_STATS = "../../data/france/stats/"

In [4]:
df, df_confirmed, dates, df_new, df_tests, df_deconf, df_sursaud, df_incid, df_tests_viros = data.import_data()

  0%|          | 0/8 [00:00<?, ?it/s]/Users/guillaumerozier/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3249: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
36it [01:50,  3.31s/it]                      

In [5]:
df_incid_fra_clage = data.import_data_tests_sexe()
df_incid_fra = df_incid_fra_clage[df_incid_fra_clage["cl_age90"]==0]
df_france = df.groupby(["jour"]).sum().reset_index()

In [6]:
departements = departements = list(dict.fromkeys(list(df_incid['dep'].values))) 
regions = departements = list(dict.fromkeys(list(df_incid['regionName'].dropna().values))) 

df_regions = df.groupby(["jour", "regionName"]).sum().reset_index()
df_incid_regions = df_incid.groupby(["jour", "regionName"]).sum().reset_index()

In [17]:
def generate_data(data_incid, data_hosp):## Incidence
    dict_data = {}

    taux_incidence = data_incid["P"].rolling(window=7).sum().fillna(0) * 100000 / data_incid["pop"].values[0]
    dict_data["incidence"] = {"jour": list(data_incid.jour), "valeur": list(taux_incidence)}
    
    taux_positivite = (data_incid["P"].rolling(window=7).sum() * 100 / data_incid["T"].rolling(window=7).sum()).fillna(0)
    dict_data["taux_positivite"] = {"jour": list(data_incid.jour), "valeur": list(taux_positivite)}

    cas = data_incid["P"].rolling(window=7).mean().fillna(0)
    dict_data["cas"] = {"jour": list(data_incid.jour), "valeur": list(cas)}

    hospitalisations = data_hosp.hosp.fillna(0)
    dict_data["hospitalisations"] = {"jour": list(data_hosp.jour), "valeur": list(hospitalisations)}

    reanimations = data_hosp.rea.fillna(0)
    dict_data["reanimations"] = {"jour": list(data_hosp.jour), "valeur": list(reanimations)}

    deces_hospitaliers = data_hosp.dc.diff().rolling(window=7).mean().fillna(0)
    dict_data["deces_hospitaliers"] = {"jour": list(data_hosp.jour), "valeur": list(deces_hospitaliers)}
    
    return dict_data
 

In [18]:
def export_data(data):
    with open(PATH_STATS + 'dataexplorer.json', 'w') as outfile:
        json.dump(data, outfile)

In [19]:
def dataexplorer():
    dict_data = {}
    dict_data["regions"] = regions
    dict_data["france"] = generate_data(df_incid_fra, df_france)
    
    for reg in regions:
        dict_data[reg] = generate_data(df_incid_regions[df_incid_regions.regionName==reg], df_regions[df_regions.regionName==reg])
    
    export_data(dict_data)

In [20]:
dataexplorer()

In [24]:
df_incid_regions[df_incid_regions.regionName==regions[0]]

,jour,regionName,P,cl_age90,pop,regionCode,T,incidence
0,2020-05-13,Auvergne-Rhône-Alpes,113,6372,16064754.0,11088.0,44561,0.000000
18,2020-05-14,Auvergne-Rhône-Alpes,206,6372,16064754.0,11088.0,52888,0.000000
36,2020-05-15,Auvergne-Rhône-Alpes,156,6372,16064754.0,11088.0,53020,0.000000
54,2020-05-16,Auvergne-Rhône-Alpes,50,6372,16064754.0,11088.0,22011,0.000000
72,2020-05-17,Auvergne-Rhône-Alpes,26,6372,16064754.0,11088.0,7007,0.000000
...,...,...,...,...,...,...,...,...
4716,2021-01-30,Auvergne-Rhône-Alpes,3108,6372,16064754.0,11088.0,252032,32228.746906
4734,2021-01-31,Auvergne-Rhône-Alpes,640,6372,16064754.0,11088.0,38797,32141.886660
4752,2021-02-01,Auvergne-Rhône-Alpes,8056,6372,16064754.0,11088.0,529133,31756.364573
4770,2021-02-02,Auvergne-Rhône-Alpes,5712,6372,16064754.0,11088.0,442035,31625.293784
